In [ ]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

In [ ]:
import sys

sys.path.append('../scripts')

In [ ]:
import numpy as np
import scanpy as sc
import anndata as ad
import os
import seaborn as sns
import matplotlib.pyplot as plt

from tqdm import tqdm
from sklearn.model_selection import train_test_split

from cellina import CellinaModel
from utils import set_seed
from train_loo import preprocess_adata

In [ ]:
plt.rcParams['font.family'] = 'monospace'
plt.rcParams['font.size'] = 16
plt.rcParams['figure.dpi'] = 150

In [ ]:
import cellina
cellina.__version__

# Get dataset

In [ ]:
set_seed(0)

In [ ]:
slide_id = 'crc_120'

In [ ]:
data_path = f"/data/a330d/datasets/crc/raw_zenodo/{slide_id}.h5ad"
adata = sc.read(data_path)
adata.obs_names_make_unique()

In [ ]:
adata = preprocess_adata(adata, n_neighbors=200)

In [ ]:
adata

In [ ]:
labels_key = 'coarse_type'
domains_key = 'typ_clean'
batch_key = 'sid'

In [ ]:
fig_save_path = "../figures/application"

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))  # width x height in inches

sc.pl.spatial(
    adata,
    color=domains_key,
    ax=ax,
    spot_size=100,
    title=f"Slide 210",
    show=False,
)
# Remove x and y axis labels
ax.set_xlabel('')
ax.set_ylabel('')

# Optionally, remove ticks too
ax.set_xticks([])
ax.set_yticks([])
plt.show()

fig.savefig(f"{fig_save_path}/{slide_id}slide.png", bbox_inches='tight')

## Data splits

In [ ]:
split = "random"

# Get holdout indices
if split == "random":
    fraction = 0.1
    n_cells = adata.n_obs
    n_holdout = int(n_cells * fraction)

    # Randomly choose cells
    test_idx = np.random.choice(n_cells, n_holdout, replace=False)

elif split == "ood":
    holdout_ct = "Fibroblast"
    is_tumor_region  = adata.obs[domains_key].str.contains("CRC", regex=True)
    is_holdout_ct = adata.obs[labels_key] == holdout_ct

    # Combine for test set
    test_mask = (is_tumor_region) & (is_holdout_ct)
    test_idx = np.where(test_mask)[0]
else:
    raise ValueError(f"Unknown split: {split}")

# Get train/val indices
all_idx = np.arange(adata.n_obs)
trainval_idx = np.setdiff1d(all_idx, test_idx)

In [ ]:
# Set 'is_holdout' to False by default, then True for selected cells
adata.obs['is_holdout'] = False
adata.obs.iloc[test_idx, adata.obs.columns.get_loc('is_holdout')] = True

In [ ]:
validation_size = 0.1
train_idx, val_idx = train_test_split(
    trainval_idx,
    test_size=validation_size,
    random_state=0,
    shuffle=True,
)

# Train

In [ ]:
model_base_path = f"/data/a330d/data/cellina-reproducibility/application/{slide_id}"

In [ ]:
from scvi.train._callbacks import SaveCheckpoint, EarlyStopping

model_args = {
    'adata': adata,
    'n_latent': 64,
    'n_layers': 3,
    'use_observed_lib_size': True,
    'condition_on_intrinsic': False,
    'gene_likelihood': 'nb',
    'classifier_lambda': 1.,
    'discriminator_lambda': 1.,
    }
train_args = {'max_epochs': 100,
              'batch_size': 4096,
              'check_val_every_n_epoch': 1,
              'early_stopping': True,
              'devices': [0],
              'datasplitter_kwargs': {
                  "external_indexing": [train_idx, val_idx, test_idx],
                  },
              'enable_checkpointing':True,
              'callbacks': [
                  SaveCheckpoint(
                      monitor='vae_loss_validation',
                      dirpath=f"{model_base_path}",
                      load_best_on_end=True,
                      ),
                  EarlyStopping(
                      monitor="vae_loss_validation",
                      patience=5,
                      mode="min",
                    ),
                ],
    }

plan_kwargs = {
    'lr': 1e-3,
    'normalize_losses': True
    }

In [ ]:
CellinaModel.setup_anndata(adata,
                           batch_key=batch_key,
                           labels_key=labels_key, 
                           domains_key=domains_key, 
                           spatial_obsm_key="spatial_x",
                           layer='counts')

In [ ]:
model = CellinaModel(
    **model_args, 
)
model.train(
    **train_args,
    plan_kwargs=plan_kwargs
)

# Inference and clustering

In [ ]:
checkpoint_name = os.listdir(f"{model_base_path}")[0]
model = CellinaModel.load(
    f"{model_base_path}/{checkpoint_name}",
    adata=adata,
)

In [ ]:
adata.obsm['cellina_basal'] = model.get_latent_representation(adata=adata, latent_key='z', batch_size=4096)
adata.obsm['cellina_spatial'] = model.get_latent_representation(adata=adata, latent_key='s', batch_size=4096)
adata.obsm['cellina_latent'] = model.get_latent_representation(adata=adata, batch_size=4096)

### Latent visualization

In [ ]:
x = 0.5  # fraction of cells to keep (e.g., 10%)

n_cells = adata.n_obs
n_subsample = int(n_cells * x)

# Randomly choose cell indices
np.random.seed(42)  # for reproducibility
subsample_idx = np.random.choice(n_cells, n_subsample, replace=False)

# Create the subsampled AnnData
adata_sub = adata[subsample_idx].copy()

In [ ]:
sc.pp.neighbors(adata_sub, use_rep='cellina_basal')
sc.tl.umap(adata_sub)

In [ ]:
ax = sc.pl.umap(adata_sub, color=[labels_key], wspace=0.3, show=False, title="cellina basal (z)")
# Remove x and y axis labels
ax.set_xlabel('')
ax.set_ylabel('')

# Optionally, remove ticks too
ax.set_xticks([])
ax.set_yticks([])

fig = ax.figure
plt.show()

fig.savefig(f"{fig_save_path}/crc_210_umap_basal_ct.png", bbox_inches='tight')

In [ ]:
ax = sc.pl.umap(adata_sub, color=[domains_key], wspace=0.3, show=False, title="cellina basal (z)")
# Remove x and y axis labels
ax.set_xlabel('')
ax.set_ylabel('')

# Optionally, remove ticks too
ax.set_xticks([])
ax.set_yticks([])

fig = ax.figure
plt.show()

fig.savefig(f"{fig_save_path}/crc_210_umap_basal_niche.png", bbox_inches='tight')

In [ ]:
sc.pp.neighbors(adata_sub, use_rep='cellina_spatial')
sc.tl.umap(adata_sub)

In [ ]:
ax = sc.pl.umap(adata_sub, color=[labels_key], wspace=0.3, show=False, title="cellina spatial (s)")
# Remove x and y axis labels
ax.set_xlabel('')
ax.set_ylabel('')

# Optionally, remove ticks too
ax.set_xticks([])
ax.set_yticks([])

fig = ax.figure
plt.show()

fig.savefig(f"{fig_save_path}/crc_210_umap_spatial_ct.png", bbox_inches='tight')

In [ ]:
palette = {
    # Control
    'REF': '#E69F00',

    # CRC
    'CRC': "#0075D5",

    # TVA
    'TVA': "#00B221",
}

In [ ]:
ax = sc.pl.umap(adata_sub, color=[domains_key], wspace=0.3, show=False, title="cellina spatial (s)", palette=palette)
# Remove x and y axis labels
ax.set_xlabel('')
ax.set_ylabel('')

# Optionally, remove ticks too
ax.set_xticks([])
ax.set_yticks([])

fig = ax.figure
plt.show()

fig.savefig(f"{fig_save_path}/crc_210_umap_spatial_niche.png", bbox_inches='tight')

## Leiden clusters

In [ ]:
sc.pp.neighbors(adata, n_neighbors=50, use_rep='cellina_spatial')

In [ ]:
adata_crc = adata[adata.obs['typ_clean'].str.contains('CRC')]

In [ ]:
sc.tl.leiden(adata_crc, resolution=0.35, flavor="igraph", n_iterations=-1, random_state=0)

In [ ]:
adata_crc.obs.leiden.value_counts()

In [ ]:
# remove cells with microenvironment having less then 100 cells
microenv_counts = adata_crc.obs['leiden'].value_counts()
microenvs_to_keep = microenv_counts[microenv_counts >= 100].index
adata_crc = adata_crc[adata_crc.obs['leiden'].isin(microenvs_to_keep)]

In [ ]:
# leiden resolution 0.25
sc.pl.umap(adata_crc, color=['leiden', labels_key], wspace=0.3)

In [ ]:
sc.pl.spatial(
    adata_crc,
    color=["leiden", "coarse_type", "ist"],
    palette=None,
    spot_size=100,
    show=True
)

In [ ]:
adata_crc.obs.loc[:,'microenvironment'] = [f"CRC{adata_crc.obs['leiden'].values[i]}" for i in range(len(adata_crc.obs))]

In [ ]:
adata.obs['microenvironment'] = adata.obs['microenvironment'].astype(str)

adata.obs.loc[adata_crc.obs_names, 'microenvironment'] = (
    'CRC' + adata_crc.obs['leiden'].astype(str)
)

In [ ]:
# fill with domains_key if na
mask = adata.obs['microenvironment'].isna()
adata.obs['microenvironment'] = pd.Categorical(
    np.where(mask, adata.obs[domains_key].astype(str), adata.obs['microenvironment'].astype(str))
)

In [ ]:
# remove cells with microenvironment having less then 100 cells
microenv_counts = adata.obs['microenvironment'].value_counts()
microenvs_to_keep = microenv_counts[microenv_counts >= 100].index
adata = adata[adata.obs['microenvironment'].isin(microenvs_to_keep)]

In [ ]:
sc.pl.spatial(adata, color=['microenvironment', 'typ_clean'], spot_size=100)

In [ ]:
sc.pl.spatial(adata, color=['microenvironment', 'typ_clean'], spot_size=100)

In [ ]:
# Define colorblind-friendly colors
# We'll use ColorBrewer compatible palette
ref_color = "#00B25C"    # blue for ref/tva
tva_color = "#2893EB"    # same as ref
crc_color = "#CA0022F4"    # solid red for typ_clean crc
# Shades of red for microenvironment CRC subtypes
crc_shades = ["#FF9999", "#FF4C4C", "#B22222"]  # crc0, crc1, crc2

# Assign in correct category order
adata.uns["typ_clean_colors"] = [
    {"REF": ref_color, "TVA": tva_color, "CRC": crc_color}[cat]
    for cat in adata.obs["typ_clean"].cat.categories
]

adata.uns["microenvironment_colors"] = [
    {
        "REF": ref_color,
        "TVA": tva_color,
        "CRC0": crc_shades[0],
        "CRC1": crc_shades[1],
        "CRC2": crc_shades[2],
    }[cat]
    for cat in adata.obs["microenvironment"].cat.categories
]

# Plot
sc.pl.spatial(
    adata,
    color=['microenvironment', 'typ_clean'],
    spot_size=100,
)

In [ ]:
# Define colorblind-friendly colors
# We'll use ColorBrewer compatible palette
ref_color = "#00B25C"    # blue for ref/tva
tva_color = "#2893EB"    # same as ref
crc_color = "#CA0022F4"    # solid red for typ_clean crc
# Shades of red for microenvironment CRC subtypes
crc_shades = ["#FF9999", "#FF4C4C", "#B22222"]  # crc0, crc1, crc2

# Assign in correct category order
adata.uns["typ_clean_colors"] = [
    {"REF": ref_color, "TVA": tva_color, "CRC": crc_color}[cat]
    for cat in adata.obs["typ_clean"].cat.categories
]

adata.uns["microenvironment_colors"] = [
    {
        "REF": ref_color,
        "TVA": tva_color,
        "CRC0": crc_shades[0],
        "CRC1": crc_shades[1],
        "CRC2": crc_shades[2],
    }[cat]
    for cat in adata.obs["microenvironment"].cat.categories
]

# Plot
sc.pl.spatial(
    adata,
    color=['microenvironment', 'typ_clean'],
    spot_size=100,
)

## Hotspot clusters

In [ ]:
import pandas as pd
import decoupler as dc
import hotspot

from plotting import plot_custom_umap

In [ ]:
adata_crc = adata[adata.obs[domains_key].str.contains('CRC')].copy()

In [ ]:
hs = hotspot.Hotspot(
    adata_crc,
    layer_key="counts",
    model='danb',
    latent_obsm_key="cellina_spatial",
    umi_counts_obs_key="nCount_RNA"
)

hs.create_knn_graph(
    weighted_graph=False, n_neighbors=30,
)

In [ ]:
hs_results = hs.compute_autocorrelations(jobs=24)

In [ ]:
# Select the genes with significant lineage autocorrelation
top_k = 1200
hs_genes = hs_results.loc[hs_results.FDR < 0.05].head(top_k).index

In [ ]:
# Compute pair-wise local correlations between these genes
load_lcz = False
base_dir = f'/data/a330d/datasets/crc/analysis/{slide_id}'
lcz_path = f'{base_dir}/hotspot_lcz_crc.csv'

if load_lcz:
    lcz = pd.read_csv(lcz_path, index_col=0)
    hs.local_correlation_z = lcz
else:
    lcz = hs.compute_local_correlations(hs_genes, jobs=24)
    os.makedirs(base_dir, exist_ok=True)
    lcz.to_csv(lcz_path)

In [ ]:
modules = hs.create_modules(
    min_gene_threshold=100, core_only=True, fdr_threshold=0.05
)

In [ ]:
module_scores = hs.calculate_module_scores()

module_scores.head()

In [ ]:
module_cols = []
for c in module_scores.columns:
    key = f"Module {c}"
    adata_crc.obs[key] = module_scores[c]
    module_cols.append(key)

In [ ]:
plot_custom_umap(
    adata_crc, subsample=0.1, use_rep='cellina_spatial', color=module_cols,frameon=False, vmin=-1, vmax=1, wspace=0.2
)

In [ ]:
plot_custom_umap(
    adata_crc, subsample=0.1, use_rep='cellina_spatial', color=['typ_clean', 'ist', 'jst'], frameon=False, vmin=-1, vmax=1, wspace=0.2
)

In [ ]:
module_scores_epi = module_scores.loc[adata_crc.obs_names]

In [ ]:
adata_crc.obsm['module_scores'] = module_scores_epi.values

In [ ]:
top_modules = module_scores.idxmax(axis=1)

# Add to adata.obs
adata_crc.obs["top_module"] = top_modules.astype(str)
adata_crc.obs["top_module"] = adata_crc.obs["top_module"].astype("category")

In [ ]:
# Plot
sc.pl.spatial(
    adata_crc,
    color=["top_module", "ist", "typ"],
    palette=None,
    spot_size=50,
    show=True
)

In [ ]:
# Plot
sc.pl.spatial(
    adata_crc,
    color=["top_module", "ist", "typ"],
    palette=None,
    spot_size=50,
    show=True
)

In [ ]:
# Plot
sc.pl.spatial(
    adata,
    color=["top_module", "ist", "typ"],
    palette=None,
    spot_size=50,
    show=True
)

In [ ]:
adata.obs['microenvironment'] = adata.obs[domains_key]
adata.obs.loc[adata_crc.obs_names, 'microenvironment'] = (
    'CRC' + adata_crc.obs['top_module'].astype(str)
)

In [ ]:
adata.obs['microenvironment'] = adata.obs['microenvironment'].astype('category')

In [ ]:
adata.obs.microenvironment.value_counts()

In [ ]:
# Plot how many cells of each microenvironment are present in each cell type
celltype_microenv_counts = adata.obs.groupby([labels_key, 'microenvironment']).size().reset_index(name='count')
plt.figure(figsize=(12, 6))
sns.barplot(data=celltype_microenv_counts, x=labels_key, y='count', hue='microenvironment')
plt.xticks(rotation=45)
plt.title("Cell type counts across microenvironments")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# In adata.obs['microenvironment'], put adata.obs['top_module'] only if adata.obs['typ_clean'] == 'CRC', else put adata.obs[domains_key]
#adata.obs['microenvironment'] = np.where(
#    adata.obs['typ_clean'] == 'CRC',
#    adata.obs['top_module'],
#    adata.obs[domains_key].astype(str)
#)

In [ ]:
# add adata.obs.microenvironment to adata_sub.obs by matching cell names
adata_sub.obs['microenvironment'] = adata_sub.obs_names.map(adata.obs['microenvironment'])

In [ ]:
palette = {
    # Control
    'REF': '#E69F00',  # orange

    # TVA
    'TVA': "#00B221",
    
    # CRC1 (violet)
    'CRC1': "#715095",

    # Immune hot - CRC2 (vermillion)
    'CRC2': "#D53C00",

    # CRC3 (green)
    'CRC3': "#00BC8A",

    # Immune cold - CRC4 (blue)
    'CRC4': '#0072B2',
}

In [ ]:
ax = sc.pl.umap(adata_sub[~adata_sub.obs['microenvironment'].isin(['CRC1', 'CRC3'])].copy(), color=['microenvironment'], title='cellina spatial (s)', show=False, palette=palette)
# Remove x and y axis labels
ax.set_xlabel('')
ax.set_ylabel('')

# Optionally, remove ticks too
ax.set_xticks([])
ax.set_yticks([])

fig = ax.figure
plt.show()

fig.savefig(f"{fig_save_path}/crc_210_umap_microenvs.png", bbox_inches='tight')

### Pathways

In [ ]:
from plotting import plot_pathway_activity

In [ ]:
# Get module x gene matrix
gene_modules = hs.modules
gene_c = hs.results[['C']]
df = gene_c.join(gene_modules.rename("Module"))
module_gene_matrix = df.pivot_table(index="Module", columns=df.index, values="C", fill_value=0)

In [ ]:
pw_progeny = dc.op.progeny(organism="human")
pw_hallmark = dc.op.hallmark(organism="human")

In [ ]:
# Progeny pathway activity
pw_acts, pw_padj = dc.mt.ulm(data=module_gene_matrix, net=pw_progeny)
plot_pathway_activity(pw_acts, pw_padj, alpha=0.05)

In [ ]:
# Hallmark pathway activity
pw_acts, pw_padj = dc.mt.ulm(data=module_gene_matrix, net=pw_hallmark)
plot_pathway_activity(pw_acts, pw_padj, alpha=0.05)

### Pathway targets

In [ ]:
sc.pp.log1p(adata_crc)
sc.tl.rank_genes_groups(adata_crc, groupby="top_module")

In [ ]:
deg_1 = sc.get.rank_genes_groups_df(adata_crc, group="1").set_index("names")

In [ ]:
dc.pl.source_targets(data=deg_1, x="weight", y="scores", net=pw_progeny, name="TGFb", top=15, figsize=(7, 6))

In [ ]:
dc.pl.source_targets(data=deg_1, x="weight", y="scores", net=pw_progeny, name="MAPK", top=15, figsize=(7, 6))

In [ ]:
deg_2 = sc.get.rank_genes_groups_df(adata_crc, group="2").set_index("names")

In [ ]:
dc.pl.source_targets(data=deg_2, x="weight", y="scores", net=pw_progeny, name="NFkB", top=15, figsize=(7, 6))

In [ ]:
dc.pl.source_targets(data=deg_2, x="weight", y="scores", net=pw_progeny, name="TNFa", top=15, figsize=(7, 6))

In [ ]:
dc.pl.source_targets(data=deg_2, x="weight", y="scores", net=pw_progeny, name="Hypoxia", top=15, figsize=(7, 6))

# Counterfactuals - CRC_global vs. CRC_microenv

In [ ]:
celltypes = ['Endothelial', 'Epithelial', 'Fibroblast', 'Myeloid', 'T_cell']
microenvironments = [m for m in adata.obs['microenvironment'].unique() if 'CRC' in m]
results = {k: adata[adata.obs[labels_key] == k] for k in celltypes}

In [ ]:
for ct in tqdm(celltypes, desc=f"Cell types"):
    # Control set is same - only target changes (either crc_all or microenv labels)
    is_tumor_region = adata.obs[domains_key].astype(str).str.contains('CRC', regex=True)
    is_celltype = adata.obs[labels_key].astype(str) == ct
    
    mask_control = ~is_tumor_region & is_celltype
    idx_control = np.where(mask_control.values)[0]

    # 1. Compute counterfactuals for global crc
    mask_target = is_tumor_region
    #mask_target = is_tumor_region & is_celltype
    idx_target = np.where(mask_target.values)[0]
    args = {
                "adata": adata,
                "indices": idx_control,
                "neighbour_indices": idx_target,
                "batch_size": 4096,
                "seed": 0,
            }
    results[ct].obsm['recon_x'] = model.get_normalized_expression(adata=results[ct], batch_size=4096, library_size=1e4)
    results[ct].uns[f'counterfactual_x_global'] = model.get_counterfactual_expression(**args, library_size=1e4)
    results[ct].uns[f'counterfactual_latents_global'] = model.get_counterfactual_latents(**args, latent_key='shifted')

    # 2. Loop over microenvironments
    for microenv in tqdm(microenvironments, desc=f"Microenvironments"):
        is_in_microenv = adata.obs['microenvironment'].astype(str).str.contains(microenv, regex=True)
        mask_target = is_in_microenv
        #mask_target = is_in_microenv & is_celltype
        idx_target = np.where(mask_target.values)[0]
        args["neighbour_indices"] = idx_target
        results[ct].uns[f'counterfactual_x_{microenv}'] = model.get_counterfactual_expression(**args, library_size=1e4)
        results[ct].uns[f'counterfactual_latents_{microenv}'] = model.get_counterfactual_latents(**args, latent_key='shifted')

In [ ]:
from scipy.stats import pearsonr, spearmanr
from counterfactual_analysis import safe_log2_fold_change

def _normalize_counts(counts, counts_per_k=1e4, eps=1e-8):
    return counts / (counts.sum(axis=1, keepdims=True) + eps) * counts_per_k


def compute_correlations(control, target, counterfactual, normalize_counts=True, deg=200):
    if normalize_counts:
        control = _normalize_counts(control)
        target = _normalize_counts(target)
        counterfactual = _normalize_counts(counterfactual)

    mean_control = np.nanmean(control, axis=0)
    mean_target = np.nanmean(target, axis=0)
    mean_cf = np.nanmean(counterfactual, axis=0)

    # compute log2 fold changes
    gt_vec = safe_log2_fold_change(mean_target, mean_control)
    cf_vec = safe_log2_fold_change(mean_cf, mean_control)

    deg_scores = np.abs(gt_vec)
    top_features = np.argsort(-deg_scores)[:deg]
    pear, _ = pearsonr(gt_vec[top_features], cf_vec[top_features])
    spear, _ = spearmanr(gt_vec[top_features], cf_vec[top_features])
    """
    # Plot scatterplot of gt vs cf log fold changes - highlight top features in a different color
    import matplotlib.pyplot as plt
    plt.scatter(gt_vec, cf_vec)
    plt.xlabel('Ground Truth Log2 Fold Change')
    plt.ylabel('Counterfactual Log2 Fold Change')
    plt.title('GT vs CF Log2 Fold Change')
    
    # highlight top features in a different color
    plt.scatter(gt_vec[top_features], cf_vec[top_features], color='red')
    plt.show()
    """
    return float(pear), float(spear)

In [ ]:
summary = []
deg = 200

for ct, dataset in tqdm(results.items(), desc="Computing correlations"):
    is_tumor_region = adata.obs[domains_key].astype(str).str.contains('CRC', regex=True)
    mask_control = ~dataset.obs[domains_key].astype(str).str.contains('CRC', regex=True)

    control = dataset.layers['counts'].todense()[mask_control]
    control = np.asarray(control)
    
    is_ct = dataset.obs[labels_key].astype(str) == ct
    #mask_target = is_tumor_region & is_ct
    mask_target = is_tumor_region
    target = adata.layers['counts'].todense()[mask_target]
    target = np.asarray(target)

    counterfactual = dataset.uns['counterfactual_x_global']
    pear_global, spear_global = compute_correlations(control, target, counterfactual, deg=deg)

    summary.append({
        "cell_type": ct,
        "label": "CRC_global",
        "pearson": np.round(pear_global, 4),
        "spearman": np.round(spear_global, 4)
    })

    for microenv in tqdm(microenvironments, desc="Microenvironments"):
        is_in_microenv = adata.obs['microenvironment'].astype(str).str.contains(microenv, regex=True)
        
        #mask_target = is_in_microenv & is_ct
        mask_target = is_in_microenv
        target = adata.layers['counts'].todense()[mask_target]

        target = np.asarray(target)
        counterfactual = dataset.uns[f'counterfactual_x_{microenv}']
        pear_microenv, spear_microenv = compute_correlations(control, target, counterfactual, deg=deg)
        summary.append({
            "cell_type": ct,
            "label": microenv,
            "pearson": np.round(pear_microenv, 4),
            "spearman": np.round(spear_microenv, 4)
        })

In [ ]:
summary_df = pd.DataFrame(summary)

In [ ]:
# Save df
summary_df.to_csv(f"../results/microenvironments_{slide_id}.csv", index=False)

# Dumbbell plots

In [ ]:
# load df
summary_df = pd.read_csv(f"../results/microenvironments_{slide_id}.csv")

In [ ]:
# Prepare data for plotting
plot_data = []
for corr_type in ["pearson","spearman"]:
    tmp = summary_df.copy()
    # Global value
    global_vals = tmp[tmp['label'] == 'CRC_global'].set_index('cell_type')[corr_type]
    # Mean of all others
    mean_others = tmp[tmp['label'] != 'CRC_global'].groupby('cell_type')[corr_type].mean()
    plot_data.append((global_vals, mean_others))

cell_types = summary_df['cell_type'].unique()

# Colorblind-friendly palette
colors = {
    'CRC_global': '#0072B2',      # blue
    'mean_others': '#D55E00'      # orange/red
}

# Monospace font
plt.rcParams['font.family'] = 'monospace'

# Plot
fig, axes = plt.subplots(1, 2, figsize=(10, 5), sharey=True)

for ax, (global_vals, mean_others), title in zip(
    axes, 
    plot_data, 
    [r"Pearson $r$", r"Spearman $\rho$"]
):
    y_pos = np.arange(len(cell_types))
    
    # Dumbbell lines
    ax.hlines(y=y_pos, xmin=mean_others.values, xmax=global_vals.values, color='gray', alpha=1, linewidth=2)
    
    # Scatter points with different shapes
    ax.scatter(global_vals.values, y_pos, color=colors['CRC_global'], s=100, marker='o', label='Global CRC')
    ax.scatter(mean_others.values, y_pos, color=colors['mean_others'], s=100, marker='D', label='Within-microenvironment')
    
    # Y-axis labels
    ax.set_yticks(y_pos)
    ax.set_yticklabels(cell_types)
    #ax.set_xticklabels(ax.get_xticks(), fontsize=10)
    
    # X-axis
    ax.set_xlim(0, 1)
    ax.set_xlabel(title, fontsize=14)

# Overall figure title
fig.suptitle("Global vs. Microenvironment-specific predictions", fontsize=16, fontweight='bold', y=0.9)


# Get legend handles
handles, labels = axes[0].get_legend_handles_labels()

# Place legend BELOW the title, centered, horizontal
fig.legend(
    handles, labels,
    loc='upper center',
    bbox_to_anchor=(0.5, 0.86),  # just below suptitle
    ncol=2,                      # horizontal layout
    frameon=False,
    fontsize=14,                  # increase text size
    markerscale=1.0
)
# Adjust subplots to make room for legend on far right
fig.subplots_adjust(top=0.88)  # shrink axes to leave 20% for legend

plt.tight_layout(rect=[0, 0, 1, 0.88])
plt.savefig(f"{fig_save_path}/dumbbell_{slide_id}.svg", bbox_inches='tight')
plt.show()

In [ ]:
tmp = summary_df.copy()

global_vals = tmp[tmp['label'] == 'CRC_global'].set_index('cell_type')['spearman']
mean_others = tmp[tmp['label'] != 'CRC_global'].groupby('cell_type')['spearman'].mean()

cell_types = summary_df['cell_type'].unique()

# Colorblind-friendly palette
colors = {
    'CRC_global': '#0072B2',
    'mean_others': '#D55E00'
}

plt.rcParams['font.family'] = 'monospace'

fig, ax = plt.subplots(1, 1, figsize=(5, 5))

y_pos = np.arange(len(cell_types))

# Dumbbell lines
ax.hlines(
    y=y_pos,
    xmin=mean_others.values,
    xmax=global_vals.values,
    color='gray',
    linewidth=2
)

# Points
ax.scatter(global_vals.values, y_pos,
           color=colors['CRC_global'], s=100, marker='o', label='CRC global')

ax.scatter(mean_others.values, y_pos,
           color=colors['mean_others'], s=100, marker='D', label='CRC subtype (mean)')

# Y-axis
ax.set_yticks(y_pos)
ax.set_yticklabels(cell_types)

# X-axis (UPDATED RANGE)
ax.set_xlim(0.2, 0.9)
ax.set_xlabel(r"Spearman $\rho$", fontsize=14)

# Title
fig.suptitle(
    "CRC global vs. subtype predictions",
    fontsize=16,
    fontweight='bold',
    y=0.92
)

# Legend
handles, labels = ax.get_legend_handles_labels()
fig.legend(
    handles, labels,
    loc='upper center',
    bbox_to_anchor=(0.5, 0.88),
    ncol=2,
    frameon=False,
    fontsize=14
)

# Layout
fig.subplots_adjust(top=0.82)
plt.tight_layout(rect=[0, 0, 1, 0.85])
plt.savefig(f"{fig_save_path}/dumbbell_spearman_{slide_id}.svg", bbox_inches='tight')
plt.show()

In [ ]:
# Read files in a loop, concatenate into a single dataframe, and plot distribution of pearson and spearman correlations across cell types for global vs microenv predictions
import glob

results_path = "../results"
all_files = glob.glob(f"{results_path}/microenvironments_*.csv")
dfs = []
for file in all_files:
    df = pd.read_csv(file)
    df['slide_id'] = file.split("_")[-1].split(".")[0]  # extract slide_id from filename
    dfs.append(df)
dumbbell_df = pd.concat(dfs, ignore_index=True)

In [ ]:
tmp = dumbbell_df.copy()

# Create a new column grouping labels
tmp['group'] = np.where(
    tmp['label'] == 'CRC_global',
    'CRC global',
    'CRC subtype'
)

# Keep only needed columns
plot_df = tmp[['cell_type', 'group', 'spearman']]

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

sns.boxplot(
    data=plot_df,
    x='spearman',
    y='cell_type',
    hue='group',
    palette={
        'CRC global': '#0072B2',
        'CRC subtype': '#D55E00'
    },
    ax=ax
)

ax.set_xlim(0.2, 1.)
ax.set_xlabel(r"Spearman $\rho$", fontsize=14)

fig.suptitle(
    "CRC global vs. subtype predictions",
    fontsize=16,
    fontweight='bold',
    y=0.95
)

# Legend on top (like before)
handles, labels = ax.get_legend_handles_labels()
fig.legend(
    handles, labels,
    loc='upper center',
    bbox_to_anchor=(0.5, 0.9),
    ncol=2,
    frameon=False,
    fontsize=14
)

ax.get_legend().remove()

plt.tight_layout(rect=[0, 0, 1, 0.88])
plt.savefig(f"{fig_save_path}/boxplot_spearman.svg", bbox_inches='tight')
plt.show()

In [ ]:
def subsample_adata(adata, fraction=0.1, random_state=42):
    x = fraction

    n_cells = adata.n_obs
    n_subsample = int(n_cells * x)

    # Randomly choose cell indices
    np.random.seed(random_state)  # for reproducibility
    subsample_idx = np.random.choice(n_cells, n_subsample, replace=False)

    # Create the subsampled AnnData
    return adata[subsample_idx].copy()

In [ ]:
adata.obsm['recon_x'] = model.get_normalized_expression(adata=adata, batch_size=4096, library_size=1e4)

In [ ]:
# Get cellina spatial for non CRC cells
adata_list = []
select_ct = "Fibroblast"
adata_fib = results[select_ct].copy()
control_mask = adata_fib.obs[domains_key].astype(str).str.contains('REF')
X = adata_fib.obsm['recon_x'][control_mask.values, :]
X = np.log1p(X)
adata_control = ad.AnnData(X=X, 
                           obs=adata_fib.obs[control_mask])
#adata_list.append(subsample_adata(adata_control, fraction=0.1, random_state=0))
adata_list.append(subsample_adata(adata_control, fraction=1, random_state=0))

for microenv in microenvironments:
    # Get real latents for cells in this microenvironment
    real_mask = adata.obs['microenvironment'].astype(str).str.contains(microenv) & (adata.obs[labels_key].astype(str) == select_ct)
    X = adata.obsm['recon_x'][real_mask.values, :]
    X = np.log1p(X)
    obs=adata.obs[real_mask]
    adata_real = ad.AnnData(X=X, obs=obs)
    adata_real.obs['microenvironment'] = microenv
    #adata_list.append(subsample_adata(adata_real, fraction=0.2, random_state=0))
    adata_list.append(subsample_adata(adata_real, fraction=1., random_state=0))
    
    # Get counterfactual latents for this microenvironment
    X = adata_fib.uns[f'counterfactual_x_{microenv}']
    X = np.log1p(X)
    obs = adata_fib.obs[~adata_fib.obs[domains_key].str.contains('CRC')]
    obs['microenvironment'] = f"{microenv}_counterfactual"
    adata_cf = ad.AnnData(X=X, obs=obs)
    #adata_list.append(subsample_adata(adata_cf, fraction=0.1, random_state=0))
    adata_list.append(subsample_adata(adata_cf, fraction=1., random_state=0))

In [ ]:
# Merge all adatas in list
adata_fib_merged = ad.concat(adata_list, join='outer')
adata_fib_merged.obs_names_make_unique()

In [ ]:
adata_fib_merged.obs['microenvironment'].value_counts()

In [ ]:
sc.pp.pca(adata_fib_merged, n_comps=50)
sc.pp.neighbors(adata_fib_merged, n_pcs=50)
sc.tl.umap(adata_fib_merged)

In [ ]:
# rename REF, TVA to control
#adata_fib_merged.obs['microenvironment'] = adata_fib_merged.obs['microenvironment'].replace({'REF': 'Control', 'TVA': 'Control'})
adata_fib_merged.obs['domain'] = adata_fib_merged.obs['microenvironment']
adata_fib_merged.obs['domain'] = adata_fib_merged.obs['domain'].replace({'REF': 'Control', 'TVA': 'Control'})
adata_fib_merged.obs['domain'] = adata_fib_merged.obs['domain'].astype('category')

In [ ]:
palette = {
    # Control
    'Control': '#E69F00',  # orange

    # Immune hot - CRC2 (vermillion)
    'CRC2': "#580803",
    'CRC2_counterfactual': '#F4A582',  # lighter version

    # Immune cold - CRC4 (blue)
    'CRC4': "#011080",
    'CRC4_counterfactual': '#92C5DE',  # lighter version
}

In [ ]:
ax = sc.pl.umap(
    adata_fib_merged[
        adata_fib_merged.obs['domain'].isin([
            'Control', 'CRC2', 'CRC2_counterfactual',
            'CRC4', 'CRC4_counterfactual'
        ])
    ].copy(),
    color='domain',
    palette=palette,
    show=False,
    title=f"{select_ct}: Control → Target"
)
# Increase legend font size
legend = ax.get_legend()
if legend is not None:
    for text in legend.get_texts():
        text.set_fontsize(14)  # adjust as needed

# Rasterize only the UMAP points
for coll in ax.collections:
    coll.set_rasterized(True)

# Remove x and y axis labels
ax.set_xlabel('')
ax.set_ylabel('')

# Optionally, remove ticks too
ax.set_xticks([])
ax.set_yticks([])

ax.set_frame_on(False)

fig = ax.figure
plt.show()

#fig.savefig(f"{fig_save_path}/crc_210_umap_cf_Fibroblasts.svg", bbox_inches='tight')

# Pathway-based perturbations

In [ ]:
is_celltype = "Fibroblast"

In [ ]:
adata_pert = results[is_celltype].copy()

In [ ]:
adata_pert

In [ ]:
# rename REF, TVA to control
adata_pert.obs['domain'] = adata_pert.obs['microenvironment']
adata_pert.obs['domain'] = adata_pert.obs['domain'].replace({'REF': 'Control', 'TVA': 'Control'})
adata_pert.obs['domain'] = adata_pert.obs['domain'].astype('category')

In [ ]:
adata_pert.obs.domain.value_counts()

In [ ]:
adata_pert.X = adata_pert.layers['counts'].copy()
sc.pp.normalize_total(adata_pert, target_sum=1e4)
sc.pp.log1p(adata_pert)

1. Find DE(Ref, CRC1) and DE(Ref, CRC2)

In [ ]:
select_groups = ['CRC1', 'CRC2']
adata_pert = adata_pert[adata_pert.obs['domain'].isin(['Control'] + select_groups)].copy()

sc.tl.rank_genes_groups(
    adata_pert,
    groupby='domain',
    groups=select_groups,
    reference='Control',
    method='wilcoxon'
)

a_raw = sc.get.rank_genes_groups_df(adata_pert, group=select_groups[0])
b_raw = sc.get.rank_genes_groups_df(adata_pert, group=select_groups[1])

In [ ]:
a = a_raw[(a_raw['pvals_adj'] < 0.05) & (abs(a_raw['logfoldchanges']) > 1)]
b = b_raw[(b_raw['pvals_adj'] < 0.05) & (abs(b_raw['logfoldchanges']) > 1)]

In [ ]:
a

In [ ]:
b

2. Find pathway-defining genes

In [ ]:
pathway = "TGFb"
weight_threshold = 1
c = pw_progeny[(pw_progeny.source==pathway) & (pw_progeny.weight>weight_threshold)]

In [ ]:
pathway = "NFkB"
weight_threshold = 1
d = pw_progeny[(pw_progeny.source==pathway) & (pw_progeny.weight>weight_threshold)]

3. Find intersection of DE genes and pathway genes

In [ ]:
P = set(a.names).intersection(set(c.target))
#Q = set(b.names).intersection(set(c.target))

In [ ]:
#P = set(a.names).intersection(set(d.target))
Q = set(b.names).intersection(set(d.target))

In [ ]:
len(P), len(Q)

4. Generate counterfactual populations for each target gene set

In [ ]:
logfc_series_a = a[a.names.isin(list(P))]
logfc_series_a = logfc_series_a.set_index('names')['logfoldchanges']

logfc_set_a = {is_celltype: logfc_series_a}

In [ ]:
from cellina._spatial_utils import make_neighbor_perturbation

make_neighbor_perturbation(
    adata,
    perturbations=logfc_set_a,
    groupby=labels_key,
    obsm_key_out='spatial_x_cf',
    base=np.e,
)

In [ ]:
adata.X = adata.layers['counts'].copy()

In [ ]:
is_tumor_region = adata.obs[domains_key].astype(str).str.contains('CRC', regex=True)
mask_control = ~is_tumor_region & (adata.obs[labels_key]==is_celltype)
idx_control = np.where(mask_control.values)[0]

In [ ]:
adata_pert.uns['pert_P'] = model.get_perturbed_expression(
    adata,
    indices=idx_control,
    batch_size=4096,
    spatial_obsm_key="spatial_x_cf",
    library_size=1e4)

In [ ]:
logfc_series_b = b[b.names.isin(list(Q))]
logfc_series_b = logfc_series_b.set_index('names')['logfoldchanges']

logfc_set_b = {is_celltype: logfc_series_b}

In [ ]:
from cellina._spatial_utils import make_neighbor_perturbation

make_neighbor_perturbation(
    adata,
    perturbations=logfc_set_b,
    groupby=labels_key,
    obsm_key_out='spatial_x_cf',
    base=np.e,
)

In [ ]:
adata_pert.uns['pert_Q'] = model.get_perturbed_expression(
    adata,
    indices=idx_control,
    batch_size=4096,
    spatial_obsm_key="spatial_x_cf",
    library_size=1e4)

In [ ]:
adata_pert.obs.domain.value_counts()

5. Correlation with observed microenv sets CRC1 (pert_P) and CRC2 (pert_Q)

In [ ]:
mask_control = ~adata_pert.obs['domain'].str.contains('CRC')
control = adata_pert.layers['counts'].todense()[mask_control]
control = np.asarray(control)

In [ ]:
is_in_microenv = adata_pert.obs['domain'].str.contains('CRC1')
mask_target = is_in_microenv
target = adata_pert.layers['counts'].todense()[mask_target]
target = np.asarray(target)

counterfactual_p = adata_pert.uns['pert_P']

In [ ]:
pear_p, spear_p = compute_correlations(control, target, counterfactual_p, deg=deg)
pear_p, spear_p

In [ ]:
is_in_microenv = adata_pert.obs['domain'].str.contains('CRC2')
mask_target = is_in_microenv
target = adata_pert.layers['counts'].todense()[mask_target]
target = np.asarray(target)

counterfactual_q = adata_pert.uns['pert_Q']

In [ ]:
pear_q, spear_q = compute_correlations(control, target, counterfactual_q, deg=deg)
pear_q, spear_q

In [ ]:
pert_results = {'CRC1-perturbed': {'pearson': pear_p, 'spearman': spear_p},
                'CRC2-perturbed': {'pearson': pear_q, 'spearman': spear_q}}

In [ ]:
populations = list(pert_results.keys())
metrics = ['pearson', 'spearman']

x = np.arange(len(metrics))  # [0, 1]
width = 0.35

plt.figure(figsize=(6,4))

for i, pop in enumerate(populations):
    values = [pert_results[pop][m] for m in metrics]
    plt.bar(x + i * width, values, width, label=pop)

plt.xticks(x + width / 2, metrics)
plt.ylim(0, 1)
plt.ylabel("Score")
plt.title("Correlation scores by population")
plt.legend()
plt.show()

In [ ]:
def get_lfc_vector(control, counterfactual, normalize_counts=True, deg=200):
    if normalize_counts:
        control = _normalize_counts(control)
        counterfactual = _normalize_counts(counterfactual)

    mean_control = np.nanmean(control, axis=0)
    mean_cf = np.nanmean(counterfactual, axis=0)
    

    # compute log2 fold changes
    lfc = safe_log2_fold_change(mean_cf, mean_control)
    deg_scores = np.abs(lfc)
    top_features = np.argsort(-deg_scores)[:deg]

    return lfc, top_features

In [ ]:
lfc_results = []
deg_results = []
for subtype, cf_group in zip(select_groups, ['pert_P', 'pert_Q']):
    print(subtype, cf_group)
    counterfactual = adata_pert.uns[cf_group]
    lfc, top_features = get_lfc_vector(control, counterfactual, normalize_counts=True, deg=50)
    lfc_results.append(lfc)
    deg_results.append(top_features)

In [ ]:
len(set(deg_results[0]).intersection(set(deg_results[1])))

In [ ]:
lfc_map = a_raw.set_index('names')['logfoldchanges']

sorted_lfc_a = lfc_map.reindex(adata.var_names).values

In [ ]:
lfc_map = b_raw.set_index('names')['logfoldchanges']

sorted_lfc_b = lfc_map.reindex(adata.var_names).values

In [ ]:
#lfc_A, lfc_B = lfc_results  # unpack
lfc_A, lfc_B = sorted_lfc_a, sorted_lfc_b

plt.figure(figsize=(5,5))
plt.scatter(lfc_A, lfc_B, alpha=0.3, s=5)
plt.axhline(0, color='grey', linestyle='--')
plt.axvline(0, color='grey', linestyle='--')
plt.xlabel("logFC CRC1 vs Control")
plt.ylabel("logFC CRC2 vs Control")
plt.title("Gene-wise logFC comparison")
plt.show()

In [ ]:
# Concat counterfactuals and put them in an temp adata
counterfactual_concat = np.concatenate([adata_pert[mask_control].obsm['recon_x'],
                                        counterfactual_p, 
                                        counterfactual_q], axis=0)

obs_control = adata_pert[mask_control].obs.copy()
obs_p = adata_pert[mask_control].obs.copy()
obs_p['domain'] = 'CRC1_perturbed'
obs_q = adata_pert[mask_control].obs.copy()
obs_q['domain'] = 'CRC2_perturbed'

obs_concat = pd.concat([obs_control, obs_p, obs_q], ignore_index=True)

adata_cf_concat = ad.AnnData(X=counterfactual_concat, obs=obs_concat)

In [ ]:
sc.pp.normalize_total(adata_cf_concat, target_sum=1e4)
sc.pp.log1p(adata_cf_concat)
sc.pp.pca(adata_cf_concat, n_comps=50)
sc.pp.neighbors(adata_cf_concat, n_pcs=50)
sc.tl.umap(adata_cf_concat)

In [ ]:
sc.pl.umap(adata_cf_concat, color='domain')

In [ ]:
logfc_series_a

In [ ]:
logfc_series_b